# 03_財務_費用明細

數值加總與條件加總

任務：算出全部費用總額。；只加總業務部費用。；找出最大的一筆費用與憑證編號。

規則：金額單位為新臺幣元，是已登錄費用；本例不處理稅務與核銷資格。

先人工核對再執行。資料皆為教學模擬值。

In [ ]:
%pip -q install openpyxl numpy matplotlib

In [ ]:
from pathlib import Path
import csv
from openpyxl import load_workbook

# 本機：把 Notebook 與 Excel 放在同一資料夾。Colab：先上傳本例 Excel。
SOURCE = Path('03_財務_費用明細.xlsx')
if not SOURCE.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f'請將 {SOURCE.name} 放到目前工作目錄：{Path.cwd()}')
    files.upload()
if not SOURCE.exists():
    raise FileNotFoundError(f'尚未找到 {SOURCE.name}，請確認上傳檔名')
OUT = Path('output') / '03_財務_費用明細'
OUT.mkdir(parents=True, exist_ok=True)

def read_rows(sheet):
    data = sheet.iter_rows(values_only=True)
    headers = next(data)
    return [dict(zip(headers, row)) for row in data if any(v is not None for v in row)]

def save_csv(name, records):
    if not records:
        print('沒有符合條件的資料，未輸出 CSV')
        return
    with (OUT / name).open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)

# 來源檔只有原始值，沒有需要重算的 Excel 公式。
wb = load_workbook(SOURCE, read_only=True, data_only=True)
print('工作表：', wb.sheetnames)
rows = read_rows(wb['資料'])
print('第一筆：', rows[0])

In [ ]:
total = sum(r['金額_元'] for r in rows)
sales = sum(r['金額_元'] for r in rows if r['部門'] == '業務')
largest = max(rows, key=lambda r: r['金額_元'])
print('總額：', total, '業務部：', sales, '最大筆：', largest)
assert total == 7050 and sales == 1500 and largest['憑證編號'] == 'F003'
wb.close()

## 驗收答案

總額 7,050 元；業務部 1,500 元；最大筆 F003，3,000 元。